# RF prediction test from GLOBI x SinAS data

In this notebook, we use GLOBI and SINAS data aligned with GBIF ID's to perform invasion risk predictions. 

# 1. Load in libraries and data

In [44]:
import os
import gzip
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from itertools import product
from matplotlib.patches import Patch
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc, classification_report, roc_auc_score
from tqdm.auto import tqdm
from pygbif import species as gbif_species, occurrences
import time

In [24]:
# ==============================================================================
# 1. CONFIGURATION & DATA LOADING
# ==============================================================================
repo_root = Path.cwd().parent
sinas_path = repo_root / 'data' / 'sinas_matched_species.csv'
network_path = repo_root / 'data' / 'matched_globi_network.csv'
country_col = 'location'

print("📖 Loading datasets...")
df_sinas = pd.read_csv(sinas_path)
df_net = pd.read_csv(network_path)

# Ensure all IDs are strict integers to guarantee perfect dictionary matching
df_sinas = df_sinas.dropna(subset=['gbif_id']).copy()
df_sinas['gbif_id'] = df_sinas['gbif_id'].astype(int)
df_sinas['gbif_genus_id'] = df_sinas['gbif_genus_id'].fillna(-1).astype(int)
df_sinas['gbif_family_id'] = df_sinas['gbif_family_id'].fillna(-1).astype(int)

# Differentiate true native ranges from introduced invasion frontiers
df_sinas['is_native'] = df_sinas['establishmentMeans'].fillna('').str.lower().str.contains('native')
df_sinas['is_alien']  = df_sinas['establishmentMeans'].fillna('').str.lower().str.contains('introduced|invasive|naturalised|alien')

# Map species IDs to their higher taxonomic groups
sp_to_genus = df_sinas.set_index('gbif_id')['gbif_genus_id'].to_dict()
sp_to_family = df_sinas.set_index('gbif_id')['gbif_family_id'].to_dict()

print("✅ Datasets loaded and preprocessed successfully.")

📖 Loading datasets...
✅ Datasets loaded and preprocessed successfully.


# 2. Generate feature dataset

In [25]:
# ==============================================================================
# 2. TOTAL ONTOLOGY LEXICON DEFINITION & INTERACTION PARSING
# ==============================================================================
CONSUMPTION_TERMS = {
    'eats', 'preysOn', 'parasitizes', 'pathogenOf', 'parasitoidOf', 'endorparasitoidOf', 
    'ectoparasitoidOf', 'endoparasiteOf', 'ectoparasiteOf', 'scavenges', 'browses', 
    'grazesOn', 'hasHost', 'kills', 'saprophyteOf', 'hyperparasiteOf', 'hyperparasitoidOf', 
    'bloodFeedsOn', 'nectarFeedsOn', 'pollenFeedsOn', 'seedEaterOf', 'woodBorerOf', 
    'leafMinerOf', 'gallMakerOf', 'defoliatorOf', 'rootFeederOf', 'vectorOf', 'consumes'
}

MUTUALISM_TERMS = {
    'pollinates', 'visitsFlowersOf', 'symbiontOf', 'mutualistOf', 'hasSymbiont', 
    'commensalistOf', 'hasCommensalist', 'epiphyteOf', 'hasEpiphyte', 'inquilineOf', 
    'hasInquiline', 'phoreticOf', 'hasPhoretic', 'mycorrhizalWith', 'dispersesSeedsOf', 
    'disperses', 'vectorFor', 'sheltersWithin', 'providesShelterTo', 'nestedIn', 
    'hostsNestedIn', 'flowersVisitedBy', 'visitedBy', 'associatedWith', 'coOccursWith'
}

COMPETITION_TERMS = {
    'competesWith', 'interferesWith', 'allelopathicTo', 'inhibits', 'displaces', 'antagonistOf'
}

VICTIM_TERMS = {
    'hostOf', 'hasParasite', 'hasPathogen', 'hasParasitoid', 'preyedOnBy', 'eatenBy', 
    'parasitizedBy', 'infectedBy', 'killedBy', 'preyedUponBy'
}

def categorize_interaction(itype: str) -> str:
    if pd.isna(itype):
        return 'mutualism'
    itype_clean = itype.strip()
    itype_lower = itype_clean.lower()
    
    if itype_clean in CONSUMPTION_TERMS:  return 'resource'
    if itype_clean in MUTUALISM_TERMS:    return 'mutualism'
    if itype_clean in COMPETITION_TERMS: return 'competition'
    if itype_clean in VICTIM_TERMS:      return 'enemy'
    
    if 'by' in itype_lower:
        if any(w in itype_lower for w in ['eat', 'prey', 'parasit', 'infect', 'kill', 'patho']):
            return 'enemy'
    if any(w in itype_lower for w in ['eat', 'prey', 'consume', 'graze', 'browse', 'scaveng', 'parasit', 'patho', 'kill', 'host']):
        return 'resource'
    if any(w in itype_lower for w in ['symbio', 'mutual', 'pollin', 'commens', 'epiphyt', 'inquil', 'phor', 'assist', 'facilitat', 'cooccur', 'associate']):
        return 'mutualism'
    if any(w in itype_lower for w in ['compet', 'interfer', 'inhib', 'displac', 'antagon', 'rival']):
        return 'competition'
    return 'mutualism'

In [27]:
# ==============================================================================
# 3. POPULATING THE 3-TIER HIERARCHICAL LOOKUP TABLES (LIGHTNING FAST)
# ==============================================================================
sp_lookups =     {'resource': defaultdict(set), 'mutualism': defaultdict(set), 'enemy': defaultdict(set), 'competition': defaultdict(set)}
genus_lookups =  {'resource': defaultdict(set), 'mutualism': defaultdict(set), 'enemy': defaultdict(set), 'competition': defaultdict(set)}
family_lookups = {'resource': defaultdict(set), 'mutualism': defaultdict(set), 'enemy': defaultdict(set), 'competition': defaultdict(set)}

print("🕸️ Pre-calculating unique ontology category mappings...")
# Massive Optimization: Only categorize the unique strings once, not millions of times
unique_types = df_net['interaction_type'].dropna().unique()
interaction_mapping = {itype: categorize_interaction(itype) for itype in unique_types}

print("🕸️ Cleaning arrays and converting to high-speed arrays...")
# Filter out NaNs upfront so we don't have to check inside the loop
df_net_clean = df_net.dropna(subset=['interaction_type', 'source_gbif_id', 'target_gbif_id'])

# Extract raw values directly into fast numpy arrays
src_arr = df_net_clean['source_gbif_id'].astype(int).to_numpy()
tgt_arr = df_net_clean['target_gbif_id'].astype(int).to_numpy()
itype_arr = df_net_clean['interaction_type'].to_numpy()

print("🕸️ Sorting the global interaction network into taxonomic tiers...")
# Using raw zip() over numpy arrays runs at pure C-level speeds
for src, tgt, itype in tqdm(zip(src_arr, tgt_arr, itype_arr), total=len(src_arr), desc="Populating 3-Tier Networks"):
    
    # Instant dictionary lookup instead of row-by-row string parsing
    category = interaction_mapping.get(itype, 'mutualism')
    
    src_genus, tgt_genus = sp_to_genus.get(src), sp_to_genus.get(tgt)
    src_family, tgt_family = sp_to_family.get(src), sp_to_family.get(tgt)
    
    # Unrolling the assignments explicitly removes the inner loop completely
    if category == 'resource':
        sp_lookups['resource'][src].add(tgt)
        sp_lookups['enemy'][tgt].add(src)
        if src_genus: genus_lookups['resource'][src_genus].add(tgt)
        if tgt_genus: genus_lookups['enemy'][tgt_genus].add(src)
        if src_family: family_lookups['resource'][src_family].add(tgt)
        if tgt_family: family_lookups['enemy'][tgt_family].add(src)
        
    elif category == 'enemy':
        sp_lookups['enemy'][src].add(tgt)
        sp_lookups['resource'][tgt].add(src)
        if src_genus: genus_lookups['enemy'][src_genus].add(tgt)
        if tgt_genus: genus_lookups['resource'][tgt_genus].add(src)
        if src_family: family_lookups['enemy'][src_family].add(tgt)
        if tgt_family: family_lookups['resource'][tgt_family].add(src)
        
    elif category == 'mutualism':
        sp_lookups['mutualism'][src].add(tgt)
        sp_lookups['mutualism'][tgt].add(src)
        if src_genus: genus_lookups['mutualism'][src_genus].add(tgt)
        if tgt_genus: genus_lookups['mutualism'][tgt_genus].add(src)
        if src_family: family_lookups['mutualism'][src_family].add(tgt)
        if tgt_family: family_lookups['mutualism'][tgt_family].add(src)
        
    elif category == 'competition':
        sp_lookups['competition'][src].add(tgt)
        sp_lookups['competition'][tgt].add(src)
        if src_genus: genus_lookups['competition'][src_genus].add(tgt)
        if tgt_genus: genus_lookups['competition'][tgt_genus].add(src)
        if src_family: family_lookups['competition'][src_family].add(tgt)
        if tgt_family: family_lookups['competition'][tgt_family].add(src)

print("✅ Taxonomy lookups compiled.")

🕸️ Pre-calculating unique ontology category mappings...
🕸️ Cleaning arrays and converting to high-speed arrays...
🕸️ Sorting the global interaction network into taxonomic tiers...


Populating 3-Tier Networks:   0%|          | 0/8622749 [00:00<?, ?it/s]

✅ Taxonomy lookups compiled.


In [28]:
# ==============================================================================
# 4. PRE-COMPUTING GEOGRAPHIC AND BIOGEOGRAPHIC VECTOR BASELINES
# ==============================================================================
print("🌍 Structuring true global Presence-Absence baselines...")
all_countries = df_sinas[country_col].dropna().unique()
all_species = df_sinas['gbif_id'].unique()

# Pre-compute full Jaccard regional similarity grid via vectorized linear algebra
presence_matrix = df_sinas.pivot_table(index=country_col, columns='gbif_id', aggfunc='size', fill_value=0).clip(upper=1)
intersection = presence_matrix.dot(presence_matrix.T)
row_sums = presence_matrix.sum(axis=1)
union = row_sums.values[:, None] + row_sums.values - intersection
jaccard_matrix = intersection / np.maximum(union, 1)

# Group native vs alien distributions into direct memory arrays per species
species_native_countries = df_sinas[df_sinas['is_native']].groupby('gbif_id')[country_col].apply(set).to_dict()
species_alien_countries  = df_sinas[df_sinas['is_alien']].groupby('gbif_id')[country_col].apply(set).to_dict()
country_presence_dict = {c: set(df_sinas[df_sinas[country_col] == c]['gbif_id'].unique()) for c in all_countries}

# Pre-calculate the macro-ecological native similarity matrix to bypass loop filters
print("⚡ Pre-calculating native-range regional similarities...")
macro_sim_lookup = {}
for sp_id in all_species:
    native_set = species_native_countries.get(sp_id, set())
    if len(native_set) == 0:
        macro_sim_lookup[sp_id] = {c: 0.0 for c in all_countries}
    else:
        macro_sim_lookup[sp_id] = jaccard_matrix[list(native_set)].max(axis=1).to_dict()

🌍 Structuring true global Presence-Absence baselines...
⚡ Pre-calculating native-range regional similarities...


In [32]:
# ==============================================================================
# 5. GENERATING THE CASCADED PRESENCE-ABSENCE MATRIX
# ==============================================================================
training_records = []
total_combinations = len(all_species) * len(all_countries)

print(f"⚙️ Running 3-tier cascade vector parsing across {total_combinations} grid cells...")
for species_id, country in tqdm(product(all_species, all_countries), total=total_combinations, desc="Building ML Dataset"):
    native_set = species_native_countries.get(species_id, set())
    alien_set  = species_alien_countries.get(species_id, set())
    
    # Structural rule: Skip native ranges. We only evaluate alien frontiers and true absences.
    if country in native_set:
        continue
        
    y_label = 1 if country in alien_set else 0
    local_species = country_presence_dict[country]
    sp_genus, sp_family = sp_to_genus.get(species_id), sp_to_family.get(species_id)
    
    training_records.append({
        'gbif_id': species_id,
        'country': country,
        'feat_regional_similarity': macro_sim_lookup[species_id].get(country, 0.0),
        
        # Exact Species Layer
        'feat_sp_resource_count':   len(sp_lookups['resource'][species_id].intersection(local_species)),
        'feat_sp_mutualism_count':  len(sp_lookups['mutualism'][species_id].intersection(local_species)),
        'feat_sp_enemy_count':      len(sp_lookups['enemy'][species_id].intersection(local_species)),
        'feat_sp_competitor_count': len(sp_lookups['competition'][species_id].intersection(local_species)),
        
        # Genus Fallback Proxy Layer
        'feat_genus_resource_count':   len(genus_lookups['resource'][sp_genus].intersection(local_species)) if sp_genus else 0,
        'feat_genus_mutualism_count':  len(genus_lookups['mutualism'][sp_genus].intersection(local_species)) if sp_genus else 0,
        'feat_genus_enemy_count':      len(genus_lookups['enemy'][sp_genus].intersection(local_species)) if sp_genus else 0,
        'feat_genus_competitor_count': len(genus_lookups['competition'][sp_genus].intersection(local_species)) if sp_genus else 0,
        
        # Family Fallback Context Layer
        'feat_family_resource_count':   len(family_lookups['resource'][sp_family].intersection(local_species)) if sp_family else 0,
        'feat_family_mutualism_count':  len(family_lookups['mutualism'][sp_family].intersection(local_species)) if sp_family else 0,
        'feat_family_enemy_count':      len(family_lookups['enemy'][sp_family].intersection(local_species)) if sp_family else 0,
        'feat_family_competitor_count': len(family_lookups['competition'][sp_family].intersection(local_species)) if sp_family else 0,
        
        'is_established': y_label
    })

df_ml_complete = pd.DataFrame(training_records)
print(f"✅ Clean Training Matrix Assembled. Shape: {df_ml_complete.shape}")

⚙️ Running 3-tier cascade vector parsing across 7670349 grid cells...


Building ML Dataset:   0%|          | 0/7670349 [00:00<?, ?it/s]

✅ Clean Training Matrix Assembled. Shape: (7456114, 16)


# 3. Train random forest classifier

In [34]:
# ==============================================================================
# 6. OPTIMIZED TRAINING WITH STRATEGIC ABSENCE DOWNSAMPLING
# ==============================================================================
print("⚖️ Balancing the presence-absence matrix...")

# Separate your rare invasion events from massive global absences
df_presences = df_ml_complete[df_ml_complete['is_established'] == 1]
df_absences  = df_ml_complete[df_ml_complete['is_established'] == 0]

num_presences = len(df_presences)
num_absences  = len(df_absences)

print(f"   Total Invasion Events (1s): {num_presences}")
print(f"   Total Global Absences  (0s): {num_absences}")

# Define a background sampling ratio (e.g., 4 absences for every 1 true presence)
# This is a gold standard approach in ecological distribution modeling
absence_ratio = 4 
target_absence_count = min(num_presences * absence_ratio, num_absences)

# Randomly sample from the massive pool of absences
df_absences_sampled = df_absences.sample(n=target_absence_count, random_state=42)

# Recombine into a lean, highly informative training matrix
df_ml_balanced = pd.concat([df_presences, df_absences_sampled]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"⚡ Compressed training grid from {len(df_ml_complete)} down to {len(df_ml_balanced)} rows!")

# Configure features and target
features = [
    'feat_regional_similarity',
    'feat_sp_resource_count', 'feat_sp_mutualism_count', 'feat_sp_enemy_count', 'feat_sp_competitor_count',
    'feat_genus_resource_count', 'feat_genus_mutualism_count', 'feat_genus_enemy_count', 'feat_genus_competitor_count',
    'feat_family_resource_count', 'feat_family_mutualism_count', 'feat_family_enemy_count', 'feat_family_competitor_count'
]

X = df_ml_balanced[features]
y = df_ml_balanced['is_established']

# Split data (stratified to maintain the clean 4:1 ratio in train and test splits)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("\n🤖 Training the Random Forest Classifier on optimized subsets...")
# Added max_samples=0.5 to build individual trees on random row fractions, multiplying training speed
model = RandomForestClassifier(
    n_estimators=150, 
    random_state=42, 
    max_depth=12, 
    max_samples=0.5, 
    class_weight='balanced', 
    n_jobs=-1
)
model.fit(X_train, y_train)

# Evaluate metrics
probs = model.predict_proba(X_test)[:, 1]
print(f"\n📊 Balanced Model ROC-AUC Performance: {roc_auc_score(y_test, probs):.2f}")
print(classification_report(y_test, model.predict(X_test)))

⚖️ Balancing the presence-absence matrix...
   Total Invasion Events (1s): 138525
   Total Global Absences  (0s): 7317589
⚡ Compressed training grid from 7456114 down to 692625 rows!

🤖 Training the Random Forest Classifier on optimized subsets...

📊 Balanced Model ROC-AUC Performance: 0.88
              precision    recall  f1-score   support

           0       0.94      0.80      0.86    110820
           1       0.50      0.80      0.61     27705

    accuracy                           0.80    138525
   macro avg       0.72      0.80      0.74    138525
weighted avg       0.85      0.80      0.81    138525



In [35]:
# ==============================================================================
# 7. GENERATING REPORT VISUALIZATIONS (ROC & FEATURE IMPORTANCE)
# ==============================================================================
print("\n🎨 Rendering performance analytics graphics...")
fpr, tpr, _ = roc_curve(y_test, probs)

# Figure 1: ROC Curve
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
ax.plot(fpr, tpr, color='darkorange', lw=2.5, label=f'Cascaded Model (AUC = {auc(fpr, tpr):.2f})')
ax.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random Guess (AUC = 0.50)')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=10, fontweight='bold', labelpad=8)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=10, fontweight='bold', labelpad=8)
ax.set_title('Receiver Operating Characteristic (ROC) Curve', fontsize=12, fontweight='bold', pad=12)
ax.legend(loc="lower right", frameon=True, facecolor='white', edgecolor='none', fontsize=9)
plt.tight_layout()
plt.savefig('report_roc_curve.png', bbox_inches='tight')
plt.close()

# Figure 2: Horizontal Feature Importance Breakdown
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=True)
clean_labels = {
    'feat_regional_similarity': 'Regional Profile Similarity (Macro-Ecology)',
    'feat_sp_resource_count': 'Species Resource Count (GloBI)', 'feat_sp_mutualism_count': 'Species Mutualism Count (GloBI)',
    'feat_sp_enemy_count': 'Species Enemy Count (GloBI)', 'feat_sp_competitor_count': 'Species Competitor Count (GloBI)',
    'feat_genus_resource_count': 'Genus Resource Proxy (GloBI)', 'feat_genus_mutualism_count': 'Genus Mutualism Proxy (GloBI)',
    'feat_genus_enemy_count': 'Genus Enemy Proxy (GloBI)', 'feat_genus_competitor_count': 'Genus Competitor Proxy (GloBI)',
    'feat_family_resource_count': 'Family Resource Context (GloBI)', 'feat_family_mutualism_count': 'Family Mutualism Context (GloBI)',
    'feat_family_enemy_count': 'Family Enemy Context (GloBI)', 'feat_family_competitor_count': 'Family Competitor Context (GloBI)'
}
labels = [clean_labels.get(idx, idx) for idx in importances.index]

colors = []
for idx in importances.index:
    if 'similarity' in idx:   colors.append('#008080') # Teal
    elif 'sp_' in idx:        colors.append('#ff7f00') # Orange
    elif 'genus_' in idx:     colors.append('#33a02c') # Green
    else:                     colors.append('#1f78b4') # Blue

fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
bars = ax.barh(range(len(importances)), importances.values, color=colors, edgecolor='none', height=0.7)

for bar in bars:
    width = bar.get_width()
    if width > 0.001:
        ax.text(width + 0.005, bar.get_y() + bar.get_height()/2, f'{width:.3f}', va='center', ha='left', fontsize=8, fontweight='bold')

ax.set_yticks(range(len(importances)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Relative Feature Importance (Gini Split Metric)', fontsize=10, fontweight='bold', labelpad=8)
ax.set_title('Taxonomic-Network Cascaded Model: Feature Importances', fontsize=12, fontweight='bold', pad=12)
ax.set_xlim(0, max(importances.values) * 1.15)
ax.grid(axis='y', linestyle='', alpha=0)
ax.grid(axis='x', linestyle='--', alpha=0.5)

legend_elements = [
    Patch(facecolor='#008080', label='Macro-Ecology & Biogeography'),
    Patch(facecolor='#ff7f00', label='Species-Level Detailed Network'),
    Patch(facecolor='#33a02c', label='Genus-Level Network Proxy'),
    Patch(facecolor='#1f78b4', label='Family-Level Network Context')
]
ax.legend(handles=legend_elements, loc='lower right', frameon=True, facecolor='white', edgecolor='none', fontsize=8)
plt.tight_layout()
plt.savefig('report_feature_importances.png', bbox_inches='tight')
plt.close()

print("💾 Performance visualization graphics saved successfully to working directory.")


🎨 Rendering performance analytics graphics...
💾 Performance visualization graphics saved successfully to working directory.


# 4. Predict for country

In [37]:
# ==============================================================================
# 8. LIVE TARGET-COUNTRY HORIZON SCREENING FUNCTION
# ==============================================================================
def predict_cascaded_invasion_risk(target_country: str, trained_model, df_sinas, jaccard_matrix, macro_sim_lookup, country_presence_dict):
    if target_country not in jaccard_matrix.index:
        raise ValueError(f"❌ Country '{target_country}' is missing from spatial index footprints.")
        
    all_global_species = set(df_sinas['gbif_id'].dropna().unique())
    present_in_target = country_presence_dict[target_country]
    
    # Isolate exclusive candidates (not native, introduced, or recorded yet inside the target country)
    candidates = list(all_global_species - present_in_target)
    print(f"\n🔮 Screening {len(candidates)} unobserved species profiles for {target_country}...")
    
    id_to_name = df_sinas.dropna(subset=['gbif_id']).set_index('gbif_id')['taxon'].to_dict()
    predict_records = []
    
    for species_id in tqdm(candidates, desc=f"Evaluating vector threat matrix for {target_country}"):
        sp_genus, sp_family = sp_to_genus.get(species_id), sp_to_family.get(species_id)
        
        predict_records.append({
            'gbif_id': species_id,
            'taxon_name': id_to_name.get(species_id, 'Unknown'),
            'feat_regional_similarity': macro_sim_lookup[species_id].get(target_country, 0.0),
            
            # Species Tier
            'feat_sp_resource_count':   len(sp_lookups['resource'][species_id].intersection(present_in_target)),
            'feat_sp_mutualism_count':  len(sp_lookups['mutualism'][species_id].intersection(present_in_target)),
            'feat_sp_enemy_count':      len(sp_lookups['enemy'][species_id].intersection(present_in_target)),
            'feat_sp_competitor_count': len(sp_lookups['competition'][species_id].intersection(present_in_target)),
            
            # Genus Tier
            'feat_genus_resource_count':   len(genus_lookups['resource'][sp_genus].intersection(present_in_target)) if sp_genus else 0,
            'feat_genus_mutualism_count':  len(genus_lookups['mutualism'][sp_genus].intersection(present_in_target)) if sp_genus else 0,
            'feat_genus_enemy_count':      len(genus_lookups['enemy'][sp_genus].intersection(present_in_target)) if sp_genus else 0,
            'feat_genus_competitor_count': len(genus_lookups['competition'][sp_genus].intersection(present_in_target)) if sp_genus else 0,
            
            # Family Tier
            'feat_family_resource_count':   len(family_lookups['resource'][sp_family].intersection(present_in_target)) if sp_family else 0,
            'feat_family_mutualism_count':  len(family_lookups['mutualism'][sp_family].intersection(present_in_target)) if sp_family else 0,
            'feat_family_enemy_count':      len(family_lookups['enemy'][sp_family].intersection(present_in_target)) if sp_family else 0,
            'feat_family_competitor_count': len(family_lookups['competition'][sp_family].intersection(present_in_target)) if sp_family else 0
        })
        
    df_predict = pd.DataFrame(predict_records)
    df_predict['ml_invasion_risk_probability'] = trained_model.predict_proba(df_predict[features])[:, 1]
    
    return df_predict.sort_values('ml_invasion_risk_probability', ascending=False).reset_index(drop=True)

In [ ]:
# ==============================================================================
# 9. EXECUTION SAMPLE RUN (includes spp already in BE due to poor Sinas coverage)
# ==============================================================================
final_ranking_df = predict_cascaded_invasion_risk(
    target_country='Belgium', 
    trained_model=model, 
    df_sinas=df_sinas, 
    jaccard_matrix=jaccard_matrix, 
    macro_sim_lookup=macro_sim_lookup, 
    country_presence_dict=country_presence_dict
)

# Output top 100 horizon species sorted by calculated biological invasion risk probabilities
final_ranking_df.head(100)


🔮 Screening 22240 unobserved species profiles for Belgium...


Evaluating vector threat matrix for Belgium:   0%|          | 0/22240 [00:00<?, ?it/s]

,gbif_id,taxon_name,feat_regional_similarity,feat_sp_resource_count,feat_sp_mutualism_count,feat_sp_enemy_count,feat_sp_competitor_count,feat_genus_resource_count,feat_genus_mutualism_count,feat_genus_enemy_count,feat_genus_competitor_count,feat_family_resource_count,feat_family_mutualism_count,feat_family_enemy_count,feat_family_competitor_count,ml_invasion_risk_probability
0,3141855,Helminthotheca echioides,0.460824,0,15,6,0,0,15,6,0,42,674,151,0,0.988347
1,5361872,Ulmus americana,0.296493,1,23,9,0,2,37,16,0,2,38,17,0,0.986337
2,5361867,Ulmus rubra,0.296493,1,11,2,0,2,37,16,0,2,38,17,0,0.984393
3,9396703,Artemisia tridentata,0.296493,0,16,3,0,2,92,14,0,42,674,151,0,0.984273
4,5402972,Eupatorium serotinum,0.234603,0,5,3,0,0,20,4,0,42,674,151,0,0.983967
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,5389028,Solidago nemoralis,0.296493,0,2,0,0,2,67,8,0,42,674,151,0,0.964569
96,10902460,Salvia rosmarinus,0.460824,0,6,5,0,0,41,9,0,7,220,70,0,0.964408
97,2991369,Rubus hispidus,0.296493,0,1,0,0,5,131,97,0,39,425,188,0,0.964266
98,2949728,Chamaecrista fasciculata,0.296493,0,4,2,0,0,6,2,0,25,447,154,0,0.964097


In [48]:
def validate_horizon_threats_gbif(risk_df, gbif_country_code, species_to_validate=100, sleep_time=0.2):
    if not gbif_country_code:
        print("⚠️ No GBIF country code provided. Skipping validation step.")
        return risk_df.head(species_to_validate)

    print(f"🔍 Live-validating top threats against GBIF for country code: '{gbif_country_code}'...")
    validated_records = []
    api_calls_made = 0
    errors_hit = 0

    with tqdm(total=species_to_validate, desc="True threats found", unit="species") as pbar:
        for _, row in risk_df.iterrows():
            if len(validated_records) >= species_to_validate:
                print(f"\n🎯 Successfully secured the Top {species_to_validate} verified horizon threats!")
                break

            if api_calls_made > 0:
                time.sleep(sleep_time)

            api_calls_made += 1
            current_sp = row['taxon_name']

            try:
                # name_backbone() is broken in this pygbif version — use name_suggest() instead
                suggestions = gbif_species.name_suggest(q=current_sp, limit=1)
                
                if suggestions:
                    taxon_key = suggestions[0].get('key')         # canonical GBIF taxon ID
                    gbif_name = suggestions[0].get('scientificName', 'No match')
                    match_type = 'SUGGEST'
                else:
                    taxon_key = None
                    gbif_name = 'No match'
                    match_type = 'NONE'

                if taxon_key:
                    res = occurrences.search(taxonKey=taxon_key, country=gbif_country_code, limit=1)
                else:
                    res = {'count': 0}

                occurrence_count = res.get('count', 0)

                if occurrence_count == 0:
                    new_row = row.to_dict()
                    new_row['gbif_live_matched_name'] = f"{gbif_name} [{match_type}]"
                    new_row['gbif_live_occurrence_count'] = occurrence_count
                    new_row['gbif_taxon_key'] = taxon_key
                    validated_records.append(new_row)
                    pbar.update(1)
                # else: silently skip false positives

                pbar.set_postfix({'API Calls': api_calls_made, 'Errors': errors_hit})

            except Exception as e:
                errors_hit += 1
                time.sleep(sleep_time * 5)

                err_row = row.to_dict()
                err_row['gbif_live_matched_name'] = f"FETCH ERROR: {str(e)[:50]}"
                err_row['gbif_live_occurrence_count'] = -1
                err_row['gbif_taxon_key'] = None
                validated_records.append(err_row)
                pbar.update(1)
                pbar.set_postfix({'API Calls': api_calls_made, 'Errors': errors_hit})

    return pd.DataFrame(validated_records).reset_index(drop=True)

In [49]:
# 1. Generate the initial machine learning priority ranking
final_ranking_df = predict_cascaded_invasion_risk(
    target_country='Belgium', 
    trained_model=model, 
    df_sinas=df_sinas, 
    jaccard_matrix=jaccard_matrix, 
    macro_sim_lookup=macro_sim_lookup, 
    country_presence_dict=country_presence_dict
)

# 2. Run the post-hoc live GBIF filter to get a flawless Top 100 list
# Use 'BE' for Belgium (GBIF requires ISO 2-letter country codes)
top_100_horizon_threats = validate_horizon_threats_gbif(
    risk_df=final_ranking_df,
    gbif_country_code='BE',
    species_to_validate=100,  # Dictates the exact length of the final output list
    sleep_time=0.15           # Politeness delay between GBIF API pings
)

# 3. Save the validated list to your repository data folder
top_100_horizon_threats.to_csv(repo_root / 'data' / 'belgium_top_100_horizon_threats.csv', index=False)

# Inspect your pristine, biosecurity-verified final output!
top_100_horizon_threats.head(20)


🔮 Screening 22240 unobserved species profiles for Belgium...


Evaluating vector threat matrix for Belgium:   0%|          | 0/22240 [00:00<?, ?it/s]

🔍 Live-validating top threats against GBIF for country code: 'BE'...


True threats found:   0%|          | 0/100 [00:00<?, ?species/s]


🎯 Successfully secured the Top 100 verified horizon threats!


,gbif_id,taxon_name,feat_regional_similarity,feat_sp_resource_count,feat_sp_mutualism_count,feat_sp_enemy_count,feat_sp_competitor_count,feat_genus_resource_count,feat_genus_mutualism_count,feat_genus_enemy_count,feat_genus_competitor_count,feat_family_resource_count,feat_family_mutualism_count,feat_family_enemy_count,feat_family_competitor_count,ml_invasion_risk_probability,gbif_live_matched_name,gbif_live_occurrence_count,gbif_taxon_key
0,5361872,Ulmus americana,0.296493,1,23,9,0,2,37,16,0,2,38,17,0,0.986337,Ulmus americana L. [SUGGEST],0,5361872
1,5361867,Ulmus rubra,0.296493,1,11,2,0,2,37,16,0,2,38,17,0,0.984393,Ulmus rubra Muhl. [SUGGEST],0,5361867
2,9396703,Artemisia tridentata,0.296493,0,16,3,0,2,92,14,0,42,674,151,0,0.984273,Artemisia tridentata (Nutt.) W.A.Weber [SUGGEST],0,9396703
3,5402972,Eupatorium serotinum,0.234603,0,5,3,0,0,20,4,0,42,674,151,0,0.983967,Eupatorium serotinum Michx. [SUGGEST],0,5402972
4,2350580,Gambusia affinis,0.234603,1,7,4,0,1,7,6,0,2,7,8,0,0.981543,"Gambusia affinis (Baird & Girard, 1853) [SUGGEST]",0,2350580
5,2989228,Rubus pubescens,0.296493,0,15,3,0,5,131,97,0,39,425,188,0,0.979661,Rubus ×pubescens Genev. [SUGGEST],0,8106286
6,2466939,Anolis carolinensis,0.234603,1,7,4,0,2,7,11,0,2,7,11,0,0.979340,"Anolis carolinensis Voigt, 1832 [SUGGEST]",0,2466939
7,2704509,Distichlis spicata,0.296493,0,28,5,0,0,28,5,0,35,667,167,0,0.977831,Distichlis spicata (L.) Greene [SUGGEST],0,2704509
8,3103017,Ericameria nauseosa,0.296493,0,6,2,0,0,6,2,0,42,674,151,0,0.977787,Ericameria nauseosa (Pall. ex Pursh) G.L.Nesom...,0,3103017
9,3118746,Encelia farinosa,0.234603,0,6,2,0,0,6,2,0,42,674,151,0,0.976892,Encelia farinosa A.Gray ex Torr. [SUGGEST],0,3118746
